<a href="https://colab.research.google.com/github/Nick97382000/Lettura_bilanci/blob/main/Lettura%20bilanci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lettura bilanci

In [1]:
!git clone https://github.com/Nick97382000/Lettura_bilanci
!pip install pdfplumber PyPDF2

Cloning into 'Lettura_bilanci'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 47 (delta 15), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 29.83 MiB | 13.33 MiB/s, done.
Resolving deltas: 100% (15/15), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 87.3 MB/s eta 0:00:00


In [9]:
import PyPDF2
import pdfplumber
import pandas as pd

pdf_path_2= "Lettura_bilanci/deposito bilanci/Kirey - 2024.pdf"

with open(pdf_path_2, "rb") as file:
    reader = PyPDF2.PdfReader(file)  # Read the PDF
    text = ""
    for page in reader.pages:
        text += page.extract_text() + ""

print(text[:100])

with pdfplumber.open(pdf_path_2) as pdf:
    page = pdf.pages[2]
    tables = page.extract_tables()

    if tables:
        t = tables[0]
        print(t.shape)
        if t:
            print("KO")
        t_clean = [row for row in t if any(cell is not None for cell in row)]
        max_cols = max(len(row) for row in t_clean)
        t_uniform = [row + [None] * (max_cols - len(row)) for row in t_clean]

        df= pd.DataFrame(t_uniform)
        print(df.shape)
        print(df)




In [11]:
import pdfplumber
import pandas as pd

def trova_tabella_bilancio(pdf_path, keywords = ["immobilizzazioni immateriali",
                                                 "crediti",
                                                 "immobilizzazioni materiali", "immobilizzazioni finanziarie"]):
    with pdfplumber.open(pdf_path) as pdf:
        pagina_start = None
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for ii, t in enumerate(tables):
                testo = " ".join(
                    str(cell).lower()
                    for row in t
                    for cell in row
                    if cell
                )
                if all(kw in testo for kw in keywords):
                    pagina_start = i
                    break;
            if pagina_start is not None:
                break

        if pagina_start is None:
            return None

        tutte_le_righe = []
        intestazione = None

        for i in range (pagina_start, len(pdf.pages)):
                tables = pdf.pages[i].extract_tables()

                if not tables:
                    break

                for t in tables:
                    righe = [r for r in t if any(c for c in r)]
                    tutte_le_righe.extend(righe)

        if tutte_le_righe:
            n_colonne = max(len(r) for r in tutte_le_righe)
            tutte_le_righe = [r + [None]*(n_colonne - len(r)) for r in tutte_le_righe]

            df = pd.DataFrame(tutte_le_righe)
            return df
    return None


pdf_path_2= "Lettura_bilanci/deposito bilanci/Conserve Italia - Civilistico e Consolidato - 2024.pdf"
tabella = trova_tabella_bilancio(pdf_path_2)

print(tabella)

                                                     0  \
0                          STATO PATRIMONIALE - ATTIVO   
1                                                   A)   
2                                                        
3                                                        
4                                                   B)   
..                                                 ...   
299                                                      
300                                                      
301                                                      
302                                                      
303  Stato patrimoniale\nConto economico\nRendicont...   

                                                     1  \
0                                                 None   
1     CREDITI VERSO SOCI PER VERSAMENTI ANCORA DOVUTI:   
2    Quote capitale sociale sottoscritte e non versate   
3    TOTALE CREDITI V/SOCI PER VERSAMENTI ANCORA DO...   
4            

In [22]:
def summary_tabella(tabella):
    labels = [
        "Totale immobilizzazioni immateriali",
        "Totale immobilizzazioni materiali",
        "Totale partecipazioni",
        "Totale crediti",
        "Totale rimanenze",


    ]

    righe = {}

    if tabella.shape[1] == 3:
        col_label = 0
        cols_valori = [1, 2]
        date = tabella.iloc[0, 1:3].tolist()

    elif tabella.shape[1] == 5:
        col_label = 1
        cols_valori = [2, 3]
        date = tabella.iloc[0, 2:4].tolist()

    else:
        return pd.DataFrame()

    for label in labels:
        mask = tabella.iloc[:, col_label].astype(str).str.contains(
            label,
            case=False,
            na=False,
            regex=False
        )

        if mask.any():
            riga = tabella.loc[mask, cols_valori].iloc[0].tolist()
            righe[label] = riga

    return pd.DataFrame.from_dict(righe, orient="index", columns=date)


In [23]:
import glob
import os
pdf_files = glob.glob("Lettura_bilanci/deposito bilanci/*.pdf")

risultati = {}
summary = {}

for pdf_path in pdf_files:
    nome = os.path.basename(pdf_path).replace(".pdf", "")
    tabella = trova_tabella_bilancio(pdf_path)
    if tabella is not None:
      summary[nome]=summary_tabella(tabella)
print(summary)

{'Conserve Italia - Civilistico e Consolidato - 2024':                                     30 giugno 2025 30 giugno 2024
Totale immobilizzazioni immateriali     44.577.002     47.325.342
Totale immobilizzazioni materiali      342.717.358    306.685.238
Totale partecipazioni                   82.120.549     84.005.349
Totale crediti                                   -              -
Totale rimanenze                       205.612.371    201.996.264, 'Valtellina - 2024':                                       31-12-2024  31-12-2023
Totale immobilizzazioni immateriali    3.481.091   3.879.906
Totale immobilizzazioni materiali     20.852.418  22.781.852
Totale partecipazioni                 12.514.061   3.065.637
Totale crediti                         2.250.330   2.175.929
Totale rimanenze                     110.566.346  91.968.335, 'Gollinucci - 2024':                                      31-12-2024 31-12-2023
Totale immobilizzazioni immateriali  16.482.492    249.078
Totale immobilizzazio

In [24]:
print(summary["Conserve Italia - Civilistico e Consolidato - 2024"])

                                    30 giugno 2025 30 giugno 2024
Totale immobilizzazioni immateriali     44.577.002     47.325.342
Totale immobilizzazioni materiali      342.717.358    306.685.238
Totale partecipazioni                   82.120.549     84.005.349
Totale crediti                                   -              -
Totale rimanenze                       205.612.371    201.996.264
